# Catboost Algorithm

catboost is a stat-of-the-art machine learning algorithm that is used for classification and regression tasks. It is a gradient boosting algorithm that is based on decision trees. It is also a lazy learning algorithm, which means that it does not build a model until it is needed. It is also a lazy learning algorithm, which means that it does not build a model until it is needed. It is also a lazy learning algorithm, which means that it does not build a model until it is needed. 

In [2]:
! pip install catboost -q

In [1]:
# mport laibraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [15]:
# data import titanic
df = sns.load_dataset('titanic')
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


# pre-processing

In [16]:
# impute missing values using knn imputer in fare, age, emarked and embarktown columns
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=5)
df[['fare', 'age']] = imputer.fit_transform(df[['fare', 'age']])

# impute missing values in embarked and embarktown with mode using simple imputer
from sklearn.impute import SimpleImputer
imputer_mode = SimpleImputer(strategy='most_frequent')
df[['embarked', 'embark_town']] = imputer_mode.fit_transform(df[['embarked', 'embark_town']])
#drop deck column
df.drop('deck', axis=1, inplace=True)

# df missing values
df.isnull().sum().sort_values(ascending=False)


survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64

In [17]:
# convert each categorical column to category dtype
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

# add this as new column in the dataframe
df[categorical_cols] = df[categorical_cols].astype('category')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    category
 3   age          891 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     891 non-null    category
 8   class        891 non-null    category
 9   who          891 non-null    category
 10  adult_male   891 non-null    bool    
 11  embark_town  891 non-null    category
 12  alive        891 non-null    category
 13  alone        891 non-null    bool    
dtypes: bool(2), category(6), float64(2), int64(4)
memory usage: 49.6 KB


In [18]:
# split data into X and y
X = df.drop('survived', axis=1)
y = df['survived']

# split data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)   


In [20]:
# run the catboost classifier
model = CatBoostClassifier(iterations=100, learning_rate=0.1, 
                           depth=3, loss_function='Logloss',eval_metric='Accuracy',
                            random_seed=42, verbose=False)

#train the model
model.fit(X_train, y_train, cat_features=categorical_cols.to_list())

# predictions
y_pred = model.predict(X_test)

# evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       105
           1       1.00      1.00      1.00        74

    accuracy                           1.00       179
   macro avg       1.00      1.00      1.00       179
weighted avg       1.00      1.00      1.00       179


Confusion Matrix:
 [[105   0]
 [  0  74]]
